In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
books_df = spark.table("workspace.default.books_simple")

In [0]:
#modify genres and authors column from string to array
def parse_genres(df, column_name):
    #remove []'
    cleaned_text = F.regexp_replace(F.col(column_name), r"[\[\]']", "")
    #split on commas
    array_column = F.split(cleaned_text, r",\s*")
    return df.withColumn(column_name, array_column)


In [0]:
#create a column with the popularity of the books, from 1 to 4
#ntile(4) split all books into 4 equal sized groups by rating_count
def add_popularity_bucket(df):
    ordering = Window.orderBy(F.col("ratings_count"))
    return df.withColumn("popularity_bucket", F.ntile(4).over(ordering))

In [0]:
def select_dim_books(df):
    return df.select("book_id", "title", "authors", "genres", "original_publication_year", "pages", "average_rating", "ratings_count", "popularity_bucket")

In [0]:
dim_books_df = parse_genres(books_df, "genres")
dim_books_df = parse_genres(dim_books_df, "authors")
dim_books_df = add_popularity_bucket(dim_books_df)
dim_books_df = select_dim_books(dim_books_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.dim_books")

DataFrame[]

In [0]:
dim_books_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_books")

print(f"Saved dim_books with {dim_books_df.count()} rows")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved dim_books with 10000 rows


In [0]:
spark.table("workspace.default.dim_books").show(3)

+-------+-------------+--------------------+--------------------+-------------------------+-----+--------------+-------------+-----------------+
|book_id|        title|             authors|              genres|original_publication_year|pages|average_rating|ratings_count|popularity_bucket|
+-------+-------------+--------------------+--------------------+-------------------------+-----+--------------+-------------+-----------------+
|   7639|درخت زیبای من|[José Mauro de Va...|[fiction, classic...|                     1968|  256|          4.36|         2716|                1|
|   8946|    The Divan|             [Hafez]|[poetry, classics...|                     1380|  566|          4.63|         2773|                1|
|   6772| رباعيات خيام|[Omar Khayyám, E....|[poetry, classics...|                     1120|  184|          4.18|         3200|                1|
+-------+-------------+--------------------+--------------------+-------------------------+-----+--------------+-------------+----

In [0]:
#create book_authors table (books can have more than one author))
#authors column is array and we want to explode it to see every book per author
book_authors_df = (dim_books_df.select("book_id", F.explode("authors").alias("author")))

book_authors_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.book_authors")

print(f"Saved book_authors with {book_authors_df.count()} rows")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved book_authors with 13298 rows


In [0]:
spark.table("workspace.default.book_authors").show(3)

+-------+-------------+
|book_id|       author|
+-------+-------------+
|     12|Veronica Roth|
|     18| J.K. Rowling|
|     18|Mary GrandPré|
+-------+-------------+
only showing top 3 rows


In [0]:
#build fact_ratings 
ratings_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/workspace/default/books_raw/ratings.csv")

print(f"read {ratings_df.count()} rows from ratings.csv")

read 5976479 rows from ratings.csv


In [0]:
#verify if there are ratings for book_id that doesnt exist in dim_books
#with left_anti (we keep only rows from rating_df that have no match in dim_books_df)
orphan_ratings = ratings_df.join(dim_books_df, on="book_id", how="left_anti")
orphan_count = orphan_ratings.count()

print(f"Found {orphan_count} ratings for books_id that doesn't exsit in dim_books")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Found 0 ratings for books_id that doesn't exsit in dim_books


In [0]:
fact_ratings_df = ratings_df.select("user_id", "book_id", "rating")

In [0]:
#verify nulls
def check_no_nulls_in_key(df, key_column):
    null_count = df.filter(df[key_column].isNull()).count()
    if null_count > 0:
        raise ValueError(f"Found {null_count} rows with a missing {key_column}")
    print(f"No nulls values found in {key_column}")

In [0]:
#testing
check_no_nulls_in_key(fact_ratings_df, "user_id")
check_no_nulls_in_key(fact_ratings_df, "book_id")

No nulls values found in user_id
No nulls values found in book_id


In [0]:
fact_ratings_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.fact_ratings")

print(f"Saved fact_ratings with {fact_ratings_df.count()} rows")

Saved fact_ratings with 5976479 rows


In [0]:
spark.table("workspace.default.fact_ratings").show(3)

+-------+-------+------+
|user_id|book_id|rating|
+-------+-------+------+
|      1|    258|     5|
|      2|   4081|     4|
|      2|    260|     5|
+-------+-------+------+
only showing top 3 rows
